# Session 2: Tooling (Optuna, H2O, and Dataiku)

## Learning Objectives

In this session, we will:

1. Understand **hyperparameter tuning** with **Optuna** and compare it to **`GridSearchCV`**
2. Use **Optuna Dashboard** to explore an optimization study in the browser
3. Run **H2O AutoML** end-to-end and compare it to a simple sklearn baseline
4. Get an intuition for **Dataiku** as an external MLOps/AutoML platform (text-only intro)

The goal is not to “use the tool blindly”, but to build intuition about when and why these tools help.

## Setup

We will use `sklearn` datasets and models for the Optuna/GridSearchCV comparison.

For H2O AutoML, the code is wrapped in `try/except` so the notebook is still runnable even if `h2o` is not installed. H2O typically requires Java.

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42

In [5]:
# Data for hyperparameter tuning demos
# (We keep it 2D so you can quickly add decision-boundary plots later if desired.)

N_SAMPLES = 1200
TEST_SIZE = 0.3

# Make the task reasonably challenging so tuning matters.
# Try changing CLASS_SEP / FLIP_Y in class.
CLASS_SEP = 0.8
FLIP_Y = 0.03

X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=2,
    class_sep=CLASS_SEP,
    flip_y=FLIP_Y,
    random_state=RANDOM_STATE,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print("Train class balance:", np.bincount(y_train))

Train size: 840, Test size: 360
Train class balance: [417 423]


## 1. Optuna for Hyperparameter Tuning (vs `GridSearchCV`)

### Quick intuition

- `GridSearchCV` tries **all** combinations from a predefined grid.
- Optuna is an **adaptive** search method: it proposes new hyperparameters based on past trial results.

Both approaches:
- evaluate each configuration with cross-validation
- aim to maximize/minimize a metric (here: `f1_score`)


In [3]:
# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Use a small baseline model family so tuning stays fast.
# We will tune RandomForest hyperparameters.


In [4]:
from sklearn.model_selection import ParameterGrid

param_grid = {
    "n_estimators": [50, 150, 300],
    "max_depth": [3, 5, None],
    "min_samples_leaf": [1, 3, 5],
}

# Count how many configs the grid explores
n_configs = len(list(ParameterGrid(param_grid)))
print(f"GridSearchCV will evaluate {n_configs} hyperparameter combinations")

GridSearchCV will evaluate 27 hyperparameter combinations


### 1.1 Baseline: GridSearchCV

We’ll tune the same hyperparameters with a small grid.

**Question 1.1:**
- Why might `GridSearchCV` get slow when you increase grid size?

**Answer (example):**
- ...

In [6]:
from sklearn.model_selection import GridSearchCV

# --- GridSearchCV run ---
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=False,
)

start = time.time()
grid_search.fit(X_train, y_train)
elapsed_grid = time.time() - start

best_params_grid = grid_search.best_params_
best_cv_f1_grid = grid_search.best_score_

# Evaluate on the hold-out test set
best_rf_grid = grid_search.best_estimator_
f1_test_grid = f1_score(y_test, best_rf_grid.predict(X_test))

print("GridSearchCV results")
print("  Best params:", best_params_grid)
print(f"  Best CV F1: {best_cv_f1_grid:.4f}")
print(f"  Test F1:   {f1_test_grid:.4f}")
print(f"  Wall time: {elapsed_grid:.1f}s")

GridSearchCV results
  Best params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 150}
  Best CV F1: 0.9458
  Test F1:   0.9189
  Wall time: 19.1s


### 1.2 Optuna run (adaptive tuning)

Optuna will search the same hyperparameters space, but instead of evaluating every grid point, it will sample promising regions based on trial outcomes.

We’ll also store the study in a **SQLite database** so you can open it with **Optuna Dashboard**.

In [7]:
# --- Optuna run ---
# If Optuna isn't installed, this cell will print installation instructions.

try:
    import optuna
except ImportError:
    optuna = None

if optuna is None:
    print("Optuna not installed. Run: pip install optuna optuna-dashboard")
else:
    from sklearn.ensemble import RandomForestClassifier

    # Storage (SQLite) so the dashboard can read it
    storage_url = "sqlite:///optuna_rf_f1.db"
    study_name = "rf_f1_classification"

    def objective(trial):
        # Sample hyperparameters
        n_estimators = trial.suggest_int("n_estimators", 50, 320, log=True)
        max_depth = trial.suggest_categorical("max_depth", [None, 3, 5, 8, 12])
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        # Mean CV F1
        scores = cross_val_score(
            model, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1
        )
        return float(np.mean(scores))

    # Create / load the study
    study = optuna.create_study(
        direction="maximize",
        study_name=study_name,
        storage=storage_url,
        load_if_exists=True,
    )

    # Optimize (keep trial count small for the lab)
    start = time.time()
    study.optimize(objective, n_trials=30, show_progress_bar=False)
    elapsed_optuna = time.time() - start

    print("Optuna results")
    print("  Best CV F1:", study.best_value)
    print("  Best params:", study.best_params)
    print(f"  Wall time: {elapsed_optuna:.1f}s")

    # Evaluate on hold-out test set
    best_params = study.best_params
    best_rf_optuna = RandomForestClassifier(
        **best_params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    best_rf_optuna.fit(X_train, y_train)
    f1_test_optuna = f1_score(y_test, best_rf_optuna.predict(X_test))

    print(f"  Test F1: {f1_test_optuna:.4f}")


[I 2026-03-30 15:53:20,093] A new study created in RDB with name: rf_f1_classification
[I 2026-03-30 15:53:20,597] Trial 0 finished with value: 0.9298051427912156 and parameters: {'n_estimators': 76, 'max_depth': 12, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.9298051427912156.
[I 2026-03-30 15:53:21,503] Trial 1 finished with value: 0.9302670131258743 and parameters: {'n_estimators': 317, 'max_depth': 8, 'min_samples_leaf': 9}. Best is trial 1 with value: 0.9302670131258743.
[I 2026-03-30 15:53:21,768] Trial 2 finished with value: 0.943850402678925 and parameters: {'n_estimators': 63, 'max_depth': None, 'min_samples_leaf': 2}. Best is trial 2 with value: 0.943850402678925.
[I 2026-03-30 15:53:22,355] Trial 3 finished with value: 0.9403450872114838 and parameters: {'n_estimators': 198, 'max_depth': 8, 'min_samples_leaf': 3}. Best is trial 2 with value: 0.943850402678925.
[I 2026-03-30 15:53:23,074] Trial 4 finished with value: 0.8918197700361843 and parameters: {'n_estimators

Optuna results
  Best CV F1: 0.9495768620893802
  Best params: {'n_estimators': 111, 'max_depth': 12, 'min_samples_leaf': 1}
  Wall time: 13.6s
  Test F1: 0.9173


**Question 1.2:**
- In this lab, why might you prefer Optuna over GridSearchCV (or vice versa)?

**Answer (example):**
- ...

In [8]:
# --- Side-by-side comparison ---

grid_results = {
    "best_cv_f1_grid": best_cv_f1_grid,
    "test_f1_grid": f1_test_grid,
    "best_params_grid": best_params_grid,
    "wall_time_grid_sec": elapsed_grid,
}

print("GridSearchCV summary:")
for k, v in grid_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}" if "f1" in k else f"  {k}: {v:.1f}")
    else:
        print(f"  {k}: {v}")

# Optuna variables exist only if Optuna was installed
try:
    if optuna is not None:
        optuna_results = {
            "best_cv_f1_optuna": study.best_value,
            "test_f1_optuna": f1_test_optuna,
            "best_params_optuna": study.best_params,
            "wall_time_optuna_sec": elapsed_optuna,
        }

        print("\nOptuna summary:")
        for k, v in optuna_results.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}" if "f1" in k else f"  {k}: {v:.1f}")
            else:
                print(f"  {k}: {v}")
except NameError:
    # Study not created
    pass

GridSearchCV summary:
  best_cv_f1_grid: 0.9458
  test_f1_grid: 0.9189
  best_params_grid: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 150}
  wall_time_grid_sec: 19.1

Optuna summary:
  best_cv_f1_optuna: 0.9496
  test_f1_optuna: 0.9173
  best_params_optuna: {'n_estimators': 111, 'max_depth': 12, 'min_samples_leaf': 1}
  wall_time_optuna_sec: 13.6


### 1.3 Explore the Optuna study with Optuna Dashboard

Optuna Dashboard is a small web app that reads your Optuna storage (e.g. SQLite) and lets you inspect:

- optimization history (best value over trials)
- hyperparameter importances
- tried parameter combinations

If you optimized with SQLite storage (we did: `optuna_rf_f1.db`), you can open the dashboard like this:

```bash
$ optuna-dashboard sqlite:///optuna_rf_f1.db
```

**In a notebook**, you can run the CLI command with a shell cell.

In [ ]:
import shutil

# Try to launch the Optuna Dashboard from inside the notebook.
# Note: the command below blocks the terminal/browser session until you stop it.

if shutil.which("optuna-dashboard") is None:
    print("`optuna-dashboard` not found. Install it with: pip install optuna-dashboard")
else:
    # If you want non-blocking behavior, run this command in a separate terminal.
    # Here we show how to run it from the notebook.
    #
    # Example command (from Optuna Dashboard docs):
    #   optuna-dashboard sqlite:///optuna_rf_f1.db
    print("Starting Optuna Dashboard... open http://localhost:8080/")
    get_ipython().system_raw("optuna-dashboard sqlite:///optuna_rf_f1.db")

Starting Optuna Dashboard... open http://localhost:8080/


## 2. H2O AutoML (AutoML in one call)

H2O AutoML is an AutoML tool that can quickly train and compare many models.

**Pros**
- Very simple API (often just `H2OAutoML().train(...)`)
- Automatically trains a range of model types
- Includes a leaderboard of models

**Cons**
- Requires Java (H2O will attempt to manage this)
- Can be memory-intensive
- Less control than a manual sklearn pipeline

We’ll:
1. Train H2O AutoML on our training split
2. Evaluate the leader model on the hold-out test split
3. Compare it to a simple baseline sklearn model

In [11]:
# --- H2O AutoML demo ---
# Wrap in try/except so the notebook stays runnable.

try:
    import h2o
    from h2o.automl import H2OAutoML
except ImportError:
    h2o = None

if h2o is None:
    print("H2O not installed. Install with: pip install h2o")
else:
    h2o.init()

    # Prepare pandas DataFrames for H2O
    feature_names = [f"X{i}" for i in range(X_train.shape[1])]

    train_df = pd.DataFrame(X_train, columns=feature_names)
    test_df = pd.DataFrame(X_test, columns=feature_names)

    train_df["target"] = y_train
    test_df["target"] = y_test

    # Convert to H2OFrame
    train_h2o = h2o.H2OFrame(train_df)
    test_h2o = h2o.H2OFrame(test_df)

    # Make target categorical
    train_h2o["target"] = train_h2o["target"].asfactor()

    # Run AutoML (time-limited for the lab)
    aml = H2OAutoML(
        max_models=12,
        seed=42,
        max_runtime_secs=60,  # keep it short for demos
    )

    aml.train(x=feature_names, y="target", training_frame=train_h2o)

    # Leaderboard
    print("\nH2O AutoML Leaderboard (top models)")
    print(aml.leaderboard.head(10))

    # Predict with the leader model
    leader = aml.leader
    pred_h2o = leader.predict(test_h2o)
    pred_df = pred_h2o.as_data_frame()

    # Identify probability column for class 1
    # Commonly H2O names them: p0, p1 for binary classification.
    if "p1" in pred_df.columns:
        y_pred_proba = pred_df["p1"].values
    else:
        # Fallback: choose the last p* column
        p_cols = [c for c in pred_df.columns if c.startswith("p")]
        if not p_cols:
            raise ValueError(f"Could not find probability columns in: {pred_df.columns.tolist()}")
        y_pred_proba = pred_df[p_cols[-1]].values

    y_pred = pred_df["predict"].values

    # Convert predictions to int if needed
    y_pred = y_pred.astype(int)

    # Evaluate with sklearn metrics
    h2o_f1 = f1_score(y_test, y_pred)

    print("\nH2O leader evaluation on test set")
    print(f"  Test F1: {h2o_f1:.4f}")

    # Compare to a simple sklearn baseline
    baseline_rf = RandomForestClassifier(n_estimators=150, max_depth=None, random_state=42, n_jobs=-1)
    baseline_rf.fit(X_train, y_train)
    baseline_f1 = f1_score(y_test, baseline_rf.predict(X_test))
    print("\nBaseline sklearn RandomForest evaluation on test set")
    print(f"  Test F1: {baseline_f1:.4f}")


Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 22.0.2+9-70, mixed mode, sharing)
  Starting server from C:\SandBox\ML 203\machine_learning_for_finance_public\.venv\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\rboui\AppData\Local\Temp\tmpngrxres4
  JVM stdout: C:\Users\rboui\AppData\Local\Temp\tmpngrxres4\h2o_rboui_started_from_python.out
  JVM stderr: C:\Users\rboui\AppData\Local\Temp\tmpngrxres4\h2o_rboui_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,Europe/Paris
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,4 months and 5 days
H2O_cluster_name:,H2O_from_python_rboui_ouvnmq
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,7.913 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
15:55:43.170: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%

H2O AutoML Leaderboard (top models)
model_id                                          auc    logloss     aucpr    mean_per_class_error      rmse        mse
GBM_grid_1_AutoML_1_20260330_155543_model_1  0.972184   0.205499  0.972806               0.065488   0.239365  0.0572954
GBM_grid_1_AutoML_1_20260330_155543_model_2  0.971234   0.201939  0.973139               0.0595098  0.230919  0.0533234
XRT_1_AutoML_1_20260330_155543               0.970183   0.354733  0.964242               0.065471   0.243324  0.0592063
DRF_1_AutoML_1_20260330_155543               0.969341   0.353528  0.965408               0.063243   0.241003  0.0580825
DeepLearning_1_

c:\SandBox\ML 203\machine_learning_for_finance_public\.venv\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"



Baseline sklearn RandomForest evaluation on test set
  Test F1: 0.9189


### Question 2.1: When does AutoML help?

After you run H2O AutoML and compare to the sklearn baseline:

- Does AutoML improve F1 on the hold-out test set?
- If it does, is the improvement large enough to justify extra complexity (Java requirement, compute time)?
- If it does *not* improve, what might be happening (search budget too small, dataset too small, metric mismatch)?

**Answer (example):**
- ...

## 3. Dataiku

Dataiku (often written **Dataiku** / **Dataiku DSS**) is an external platform for building and operationalizing machine learning workflows.

Typical concepts you’ll see in a Dataiku project:

- **Datasets & connections**: central place for data access and versioning
- **Recipes / transformations**: repeatable data prep steps
- **Experiments & model training**: run and compare models inside the platform
- **Pipelines (MLOps)**: automated, scheduled end-to-end workflows
- **Model management**: register models and track artifacts/metadata
- **Monitoring & governance**: track performance, data drift, and reproducibility


## Summary

- **Optuna** provides an adaptive way to search hyperparameters and can be compared directly to **GridSearchCV** using the same CV metric (here: `f1_score`).
- **Optuna Dashboard** makes the optimization process tangible: you can inspect tried hyperparameters and how the best score evolves.
- **H2O AutoML** gives a fast “try many models” baseline with minimal code (but requires Java).
- **Dataiku** offers platform-level MLOps concepts (datasets, recipes, pipelines, governance) that you can pair with tools like Optuna/H2O.

In [ ]:
# Extra evaluation: compare H2O leader vs a simple sklearn logistic regression
# (Only runs if the H2O block above produced y_pred / y_pred_proba.)

try:
    y_pred  # from the H2O cell
    y_pred_proba

    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    lr = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    lr.fit(X_train, y_train)
    y_test_pred_lr = lr.predict(X_test)
    y_test_proba_lr = lr.predict_proba(X_test)[:, 1]

    lr_f1 = f1_score(y_test, y_test_pred_lr)
    lr_acc = accuracy_score(y_test, y_test_pred_lr)
    lr_auc = roc_auc_score(y_test, y_test_proba_lr)

    h2o_f1 = f1_score(y_test, y_pred)
    h2o_auc = roc_auc_score(y_test, y_pred_proba)
    h2o_acc = accuracy_score(y_test, y_pred)

    print("Evaluation comparison on test set")
    print("-" * 45)
    print(f"H2O leader:             F1={h2o_f1:.4f}, Acc={h2o_acc:.4f}, ROC AUC={h2o_auc:.4f}")
    print(f"Logistic Regression:   F1={lr_f1:.4f}, Acc={lr_acc:.4f}, ROC AUC={lr_auc:.4f}")

except NameError:
    print("Skipping extra evaluation: run the H2O cell first (and ensure H2O is installed).")

### Question 2.2: Baselines vs AutoML

- Is the H2O leader better than the simple sklearn logistic regression baseline on test metrics?
- What trade-off do you notice: better performance vs less interpretability / more complexity (Java + AutoML overhead)?

**Answer (example):**
- ...

## Final Wrap-up

You now have three complementary tooling perspectives:

- **Optuna**: smarter hyperparameter search than brute-force grids
- **Optuna Dashboard**: inspect and explain what the optimizer is doing
- **H2O AutoML**: a fast way to try many model families with minimal sklearn plumbing

And a **Dataiku** mental model (datasets, recipes, pipelines, model management) to connect these tools to a production workflow.